In [5]:
import torch

t = torch.tensor([1,2,3,4,5,6])
t.argmax()

tensor(5)

In [1]:
import torch.cuda
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from dataset import TextDataset
from torch.utils.data import DataLoader
from tqdm import tqdm

from model import MiniGPT

with open("corpus.txt", "r") as f:
    text = f.read()

context_window_len = 256
batch_size = 64

dataset = TextDataset(text, context_window_len)

vocab_size = dataset.tokenizer.get_vocab_size()

dataloader = DataLoader(dataset, batch_size=batch_size)

epochs = 4

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniGPT(device, vocab_size, context_window_len).to(device)
optimizer = Adam(model.parameters())
criterion = CrossEntropyLoss()

import time


for epoch in range(epochs):
    total_loss = 0
    model.train()
    for x, y in tqdm(dataloader):

        start = time.time()
        optimizer.zero_grad()
        x = x.to(device)
        y = y.to(device)
        logits = model(x)

        logits = logits.reshape(-1, logits.shape[-1])
        y = y.reshape(-1)
        print("forward:", time.time() - start)

        start = time.time()
        loss = criterion(logits, y)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        print("backward:", time.time() - start)

    print(f"Epoch {epoch+1}. Loss = {total_loss}")


  0%|          | 0/124 [00:00<?, ?it/s]

forward: 1.2377445697784424


  1%|          | 1/124 [00:02<05:14,  2.56s/it]

backward: 1.3181164264678955
forward: 1.3112530708312988


  2%|▏         | 2/124 [00:05<05:20,  2.62s/it]

backward: 1.357914686203003


  2%|▏         | 2/124 [00:05<05:37,  2.77s/it]


KeyboardInterrupt: 